# 13 - Feature Extraction from Glacial Lakes

## Objectives
- Extract geometric, terrain, dam, temporal and depth features for each detected lake
- Compute empirical depth and volume estimates using multiple literature methods
- Build a comprehensive feature table for GLOF susceptibility modelling

## Feature categories
1. **Geometric**: area, perimeter, compactness, elongation, convexity, fractal dimension
2. **Terrain**: slope, TRI within 500 m buffer (from NB11 terrain products)
3. **Dam**: freeboard, dam height (lowest boundary point approach)
4. **Temporal growth**: linear regression of area vs. year (2017–2025)
5. **Depth & volume**: ensemble of 4 empirical area-depth relations, Andean correction
6. **Risk indicators**: potential energy, size class, risk score

## Key references
- Cook & Quincey (2015) Nat. Hazards Earth Syst. Sci.
- Huggel et al. (2002) Nat. Hazards Earth Syst. Sci.
- Yao et al. (2012) Cryosphere
- O'Connor et al. (2001) J. Geophys. Res.
- Emmer (2017) validation for Peruvian lakes

In [1]:
# --- GLOF PROJECT STANDARD SETUP ---
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
try:
    import geopandas as gpd
except ImportError:
    pass

# ── Rutas ──────────────────────────────────────────────────────────────────
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import importlib
import src.gpu_utils
importlib.reload(src.gpu_utils)

# Fix PROJ: usar proj.db de rasterio (v1.6) en vez del de pyproj (v1.4)
if 'PROJ_LIB' in os.environ:
    del os.environ['PROJ_LIB']
try:
    import rasterio as _rio
    _proj_data = Path(_rio.__file__).parent / 'proj_data'
    if _proj_data.exists():
        os.environ['PROJ_LIB'] = str(_proj_data)
    del _rio, _proj_data
except Exception:
    pass

from src.gpu_utils import GPUConfig, gpu_array, cpu_array
gpu_config = GPUConfig()
print(gpu_config)

GPU_AVAILABLE = gpu_config.has_gpu
CUPY_AVAILABLE = gpu_config.cupy_available

GPU CONFIGURATION
GPU Available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU
Device Count: 1
Memory Total: 4.0 GB
Memory Free: 4.0 GB
CUDA Version: None

Library Support:
  - CuPy:         yes
  - PyTorch CUDA: no
  - Numba CUDA:   yes


## 1. Imports

In [2]:
import json
import time

import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling

import geopandas as gpd
from shapely.geometry import Point

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from scipy.stats import linregress
from tqdm import tqdm

from config_expanded_study_areas import EXPANDED_STUDY_AREAS

print('All imports successful.')

All imports successful.


## 2. Configuration

In [3]:
DATA_DIR = project_root / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

STUDY_AREAS = list(EXPANDED_STUDY_AREAS.keys())

# CRS lookup: study area -> UTM EPSG code
CRS_LOOKUP = {
    sa: EXPANDED_STUDY_AREAS[sa]['epsg'] for sa in STUDY_AREAS
}

# Terrain buffer around each lake for slope / TRI extraction
BUFFER_DIST_M = 500

# Tropical Andes regional correction for depth estimates
# Based on Emmer (2017) and Mergili et al. (2015) validation data
ANDES_DEPTH_CORRECTION = 1.15

print(f'Study areas ({len(STUDY_AREAS)}): {STUDY_AREAS}')
print(f'Buffer for terrain extraction: {BUFFER_DIST_M} m')
print(f'Andes depth correction factor: {ANDES_DEPTH_CORRECTION}')

Study areas (15): ['cordillera_blanca', 'cordillera_vilcanota', 'cordillera_central', 'chile_andes_centrales', 'cordillera_raura', 'cordillera_urubamba', 'cordillera_huanzo', 'cordillera_huayhuash', 'ecuador_antisana', 'bolivia_cordillera_real', 'patagonia_norte', 'patagonia_sur', 'bolivia_norte', 'apolobamba', 'carabaya']
Buffer for terrain extraction: 500 m
Andes depth correction factor: 1.15


## 3. Validate Prerequisites

In [4]:
def validate_prerequisites():
    """Check that all_glacial_lakes.gpkg and UTM DEMs exist."""
    print('Validating prerequisites...')
    issues = []

    lakes_path = PROCESSED_DIR / 'lakes' / 'all_glacial_lakes.gpkg'
    if not lakes_path.exists():
        issues.append(f'Missing lake file (run NB12): {lakes_path}')
    else:
        n = len(gpd.read_file(lakes_path))
        print(f'  Lakes file OK: {n} lakes')

    for area in STUDY_AREAS:
        dem_path = INTERIM_DIR / 'dem' / f'{area}_dem_utm.tif'
        if not dem_path.exists():
            issues.append(f'Missing DEM (run NB11): {dem_path}')
        else:
            print(f'  DEM OK: {area}')

    if issues:
        print('\nMISSING:')
        for iss in issues:
            print(f'  - {iss}')
        return False

    print('\nAll prerequisites OK.')
    return True


prerequisites_ok = validate_prerequisites()

Validating prerequisites...


  Lakes file OK: 12700 lakes
  DEM OK: cordillera_blanca
  DEM OK: cordillera_vilcanota
  DEM OK: cordillera_central
  DEM OK: chile_andes_centrales
  DEM OK: cordillera_raura
  DEM OK: cordillera_urubamba
  DEM OK: cordillera_huanzo
  DEM OK: cordillera_huayhuash
  DEM OK: ecuador_antisana
  DEM OK: bolivia_cordillera_real
  DEM OK: patagonia_norte
  DEM OK: patagonia_sur
  DEM OK: bolivia_norte
  DEM OK: apolobamba
  DEM OK: carabaya

All prerequisites OK.


## 4. Load Lake Data

In [5]:
lakes_path = PROCESSED_DIR / 'lakes' / 'all_glacial_lakes.gpkg'

if lakes_path.exists():
    lakes_gdf = gpd.read_file(lakes_path)
    print(f'Loaded {len(lakes_gdf)} lakes')
    print(f'Columns  : {lakes_gdf.columns.tolist()}')
    print(f'CRS      : {lakes_gdf.crs}')
    if 'area_name' in lakes_gdf.columns:
        print(f'\nLakes per area:')
        print(lakes_gdf['area_name'].value_counts().to_string())
    elif 'study_area' in lakes_gdf.columns:
        # Backwards compatibility with older column name
        lakes_gdf['area_name'] = lakes_gdf['study_area']
        print(f'\nLakes per area (from study_area col):')
        print(lakes_gdf['area_name'].value_counts().to_string())
else:
    print(f'Lake file not found: {lakes_path}')
    print('Run notebook 12 first.')
    lakes_gdf = gpd.GeoDataFrame()

Loaded 12700 lakes
Columns  : ['area_m2', 'perimeter_m', 'lake_id', 'compactness', 'elev_mean', 'elev_min', 'elev_max', 'elev_std', 'area_name', 'year', 'scene_date', 'geometry']
CRS      : EPSG:4326

Lakes per area:
area_name
cordillera_blanca          3083
cordillera_vilcanota       2770
cordillera_central         2239
chile_andes_centrales      1559
cordillera_raura           1361
cordillera_huanzo           496
cordillera_urubamba         430
cordillera_huayhuash        278
ecuador_antisana            272
bolivia_cordillera_real     108
carabaya                     87
apolobamba                    9
patagonia_sur                 8


## 5. Geometric Features

Shape descriptors derived from the lake polygon geometry.
These are scale-independent and computationally cheap.

In [6]:
def extract_geometric_features(lakes_gdf):
    """
    Compute geometric shape descriptors for each lake polygon.

    Features computed
    -----------------
    area_m2, perimeter_m  : already in GeoDataFrame (preserved)
    compactness           : 4*pi*A / P^2  (1 = circle)
    elongation            : major / minor axis of minimum rotated rectangle
    convexity             : A / convex_hull_area  (1 = convex shape)
    fractal_dim           : 2 * log(P) / log(A)  (approximate; 1 = Euclidean)
    equivalent_diameter_m : diameter of circle with same area
    shore_dev_index       : P / (2*pi*sqrt(A/pi))  (1 = circle)

    Parameters
    ----------
    lakes_gdf : GeoDataFrame

    Returns
    -------
    GeoDataFrame with new columns appended.
    """
    print('Extracting geometric features...')
    records = []

    for _, row in tqdm(lakes_gdf.iterrows(),
                       total=len(lakes_gdf),
                       desc='  Geometric features'):
        geom = row.geometry
        nan_rec = {
            'elongation': np.nan, 'convexity': np.nan,
            'fractal_dim': np.nan, 'equivalent_diameter_m': np.nan,
            'shore_dev_index': np.nan,
        }
        try:
            area = geom.area
            perim = geom.length

            # Minimum rotated rectangle for elongation
            mrr = geom.minimum_rotated_rectangle
            coords = list(mrr.exterior.coords)
            e1 = Point(coords[0]).distance(Point(coords[1]))
            e2 = Point(coords[1]).distance(Point(coords[2]))
            major = max(e1, e2)
            minor = min(e1, e2)
            elongation = major / minor if minor > 0 else 1.0

            # Convexity
            ch = geom.convex_hull
            convexity = area / ch.area if ch.area > 0 else 1.0

            # Fractal dimension (Mandelbrot perimeter method)
            fractal_dim = (2.0 * np.log(perim) / np.log(area)
                           if area > 1 and perim > 1 else np.nan)

            # Equivalent circle diameter
            eq_diam = 2.0 * np.sqrt(area / np.pi)

            # Shore development index
            circle_perim = 2.0 * np.pi * np.sqrt(area / np.pi)
            sdi = perim / circle_perim if circle_perim > 0 else 1.0

            records.append({
                'elongation': round(float(elongation), 4),
                'convexity': round(float(convexity), 4),
                'fractal_dim': round(float(fractal_dim), 4) if not np.isnan(fractal_dim) else np.nan,
                'equivalent_diameter_m': round(float(eq_diam), 2),
                'shore_dev_index': round(float(sdi), 4),
            })
        except Exception:
            records.append(nan_rec)

    feat_df = pd.DataFrame(records, index=lakes_gdf.index)
    for col in feat_df.columns:
        lakes_gdf[col] = feat_df[col]

    print(f'  Added {len(feat_df.columns)} geometric features.')
    return lakes_gdf


if len(lakes_gdf) > 0:
    lakes_gdf = extract_geometric_features(lakes_gdf)

Extracting geometric features...


  Geometric features:   0%|          | 0/12700 [00:00<?, ?it/s]

  Geometric features:   4%|▍         | 531/12700 [00:00<00:02, 5299.05it/s]

  Geometric features:   9%|▊         | 1100/12700 [00:00<00:02, 5520.75it/s]

  Geometric features:  13%|█▎        | 1653/12700 [00:00<00:02, 5104.96it/s]

...

  Geometric features:  90%|████████▉ | 11371/12700 [00:02<00:00, 5273.04it/s]

  Geometric features:  94%|█████████▎| 11899/12700 [00:02<00:00, 5242.20it/s]

  Geometric features:  98%|█████████▊| 12424/12700 [00:02<00:00, 5241.03it/s]

  Geometric features: 100%|██████████| 12700/12700 [00:02<00:00, 5007.59it/s]

  Added 5 geometric features.


## 6. Terrain Features

Slope and Terrain Ruggedness Index (TRI) statistics within a 500 m buffer
around each lake polygon. These characterise the mass-movement hazard
from surrounding slopes (relevant for ice/rock avalanche triggers).

In [7]:
# -- GPU-accelerated terrain feature extraction --
# Read each raster ONCE per area; rasterize all lake buffers as a label array;
# compute per-lake stats in one batched pass with NaN-safe sum decomposition.
# GPU path: CuPy ndimage sums + manual mean/std.   CPU fallback: scipy.ndimage.

try:
    import cupy as cp
    import cupyx.scipy.ndimage as cp_ndimage
    _CUPY_OK = True
    print('CuPy available -- terrain stats will run on GPU.')
except ImportError:
    _CUPY_OK = False
    print('CuPy not available -- falling back to scipy.ndimage (CPU).')

from scipy import ndimage as sp_ndimage
from rasterio.features import rasterize as rio_rasterize


def _batch_raster_stats(raster_path, geoms_utm, use_gpu=True):
    import numpy as _np
    n = len(geoms_utm)
    nan_arr = _np.full(n, _np.nan, dtype=_np.float32)
    if not Path(raster_path).exists() or n == 0:
        return nan_arr.copy(), nan_arr.copy(), nan_arr.copy()
    with rasterio.open(raster_path) as src:
        data = src.read(1).astype(_np.float32)
        transform = src.transform
        nd = src.nodata
        if nd is not None:
            data[data == nd] = _np.nan
    if data.size == 0:
        return nan_arr.copy(), nan_arr.copy(), nan_arr.copy()
    shapes = [(g, int(i + 1)) for i, g in enumerate(geoms_utm)]
    label_arr = rio_rasterize(
        shapes, out_shape=data.shape, transform=transform,
        fill=0, dtype=_np.int32)
    indices = _np.arange(1, n + 1, dtype=_np.int32)
    if _CUPY_OK and use_gpu:
        try:
            dg = cp.asarray(data)
            lg = cp.asarray(label_arr)
            ig = cp.asarray(indices)
            valid = ~cp.isnan(dg)
            df = cp.where(valid, dg, 0.0)
            sums   = cp_ndimage.sum(df, lg, ig)
            sums2  = cp_ndimage.sum(df * df, lg, ig)
            counts = cp_ndimage.sum(valid.astype(cp.float32), lg, ig)
            maxs   = cp_ndimage.maximum(cp.where(valid, dg, -1e30), lg, ig)
            safe_n = cp.where(counts > 0, counts, 1.0)
            means  = sums / safe_n
            stds   = cp.sqrt(cp.maximum(sums2 / safe_n - means ** 2, 0.0))
            ok     = counts > 0
            means  = cp.where(ok, means, cp.nan)
            maxs   = cp.where(ok, maxs, cp.nan)
            stds   = cp.where(ok, stds, cp.nan)
            result = (cp.asnumpy(means).astype(_np.float32),
                      cp.asnumpy(maxs).astype(_np.float32),
                      cp.asnumpy(stds).astype(_np.float32))
            del dg, lg, ig, valid, df
            cp.get_default_memory_pool().free_all_blocks()
            return result
        except Exception as e:
            print(f'  [GPU] fell back to CPU: {e}')
    # CPU fallback: nan-safe via sum decomposition
    valid = ~_np.isnan(data)
    df = _np.where(valid, data, 0.0)
    idx = list(indices)
    sums   = _np.array(sp_ndimage.sum(df,      label_arr, idx), dtype=_np.float32)
    sums2  = _np.array(sp_ndimage.sum(df * df, label_arr, idx), dtype=_np.float32)
    counts = _np.array(sp_ndimage.sum(valid.astype(_np.float32), label_arr, idx))
    maxs   = _np.array(sp_ndimage.maximum(_np.where(valid, data, -1e30), label_arr, idx), dtype=_np.float32)
    safe_n = _np.where(counts > 0, counts, 1.0)
    means  = (sums / safe_n).astype(_np.float32)
    stds   = _np.sqrt(_np.maximum(sums2 / safe_n - means ** 2, 0.0)).astype(_np.float32)
    no_data = counts == 0
    means[no_data] = _np.nan; maxs[no_data] = _np.nan; stds[no_data] = _np.nan
    return means, maxs, stds


def extract_terrain_features(lakes_gdf):
    import numpy as _np
    t0 = time.time()
    print(f'Extracting terrain features (buffer={BUFFER_DIST_M} m, GPU={_CUPY_OK})...')
    for col in ('slope_mean', 'slope_max', 'slope_std', 'tri_mean'):
        lakes_gdf[col] = _np.nan
    area_col = 'area_name' if 'area_name' in lakes_gdf.columns else 'study_area'
    for area in STUDY_AREAS:
        area_mask_bool = lakes_gdf[area_col] == area
        n_area = area_mask_bool.sum()
        if n_area == 0:
            continue
        slope_path = INTERIM_DIR / 'terrain' / area / 'slope.tif'
        tri_path   = INTERIM_DIR / 'terrain' / area / 'tri.tif'
        if not slope_path.exists():
            print(f'  SKIP {area}: slope.tif not found')
            continue
        epsg    = CRS_LOOKUP.get(area, 32718)
        utm_crs = f'EPSG:{epsg}'
        idxs    = lakes_gdf.index[area_mask_bool].tolist()
        alks    = lakes_gdf.loc[idxs].copy()
        if str(alks.crs) != utm_crs:
            alks = alks.to_crs(utm_crs)
        buf_geoms = [g.buffer(BUFFER_DIST_M) for g in alks.geometry]
        s_means, s_maxs, s_stds = _batch_raster_stats(slope_path, buf_geoms, _CUPY_OK)
        lakes_gdf.loc[idxs, 'slope_mean'] = s_means
        lakes_gdf.loc[idxs, 'slope_max']  = s_maxs
        lakes_gdf.loc[idxs, 'slope_std']  = s_stds
        if tri_path.exists():
            t_means, _, _ = _batch_raster_stats(tri_path, buf_geoms, _CUPY_OK)
            lakes_gdf.loc[idxs, 'tri_mean'] = t_means
        print(f'  Terrain done: {area} ({n_area} lakes)')
    print(f'  Total terrain time: {(time.time()-t0)/60:.1f} min')
    return lakes_gdf


if len(lakes_gdf) > 0:
    lakes_gdf = extract_terrain_features(lakes_gdf)


CuPy available -- terrain stats will run on GPU.
Extracting terrain features (buffer=500 m, GPU=True)...


  Terrain done: cordillera_blanca (3083 lakes)


  Terrain done: cordillera_vilcanota (2770 lakes)


  Terrain done: cordillera_central (2239 lakes)


  Terrain done: chile_andes_centrales (1559 lakes)


...

  Terrain done: ecuador_antisana (272 lakes)


  Terrain done: bolivia_cordillera_real (108 lakes)


  Terrain done: patagonia_sur (8 lakes)


  Terrain done: apolobamba (9 lakes)


  Terrain done: carabaya (87 lakes)
  Total terrain time: 36.6 min


## 7. Dam Features

The dam (outlet) location is estimated as the lowest point on the lake
boundary perimeter. This is a first-order approximation; true moraine
dam mapping requires high-resolution field surveys or photogrammetry.

In [8]:
def estimate_dam_features(lakes_gdf):
    """
    Estimate dam-related features from DEM elevation along lake boundary.

    Method:
      - Sample N points along the lake perimeter
      - Minimum elevation point = estimated dam / outlet
      - dam_elevation = elevation at that point
      - freeboard = mean_lake_elevation - dam_elevation
      - dam_height = dam_elevation - elevation 100 m downstream

    Adds columns: dam_elev, freeboard, dam_height

    Returns
    -------
    GeoDataFrame with dam columns added.
    """
    print('Estimating dam features...')
    nan_rec = {'dam_elev': np.nan, 'freeboard': np.nan, 'dam_height': np.nan}
    for col, val in nan_rec.items():
        lakes_gdf[col] = val

    area_col = 'area_name' if 'area_name' in lakes_gdf.columns else 'study_area'

    for area in STUDY_AREAS:
        area_mask = lakes_gdf[area_col] == area
        if area_mask.sum() == 0:
            continue

        dem_path = INTERIM_DIR / 'dem' / f'{area}_dem_utm.tif'
        if not dem_path.exists():
            print(f'  SKIP {area}: DEM not found')
            continue

        epsg = CRS_LOOKUP.get(area, 32718)
        utm_crs = f'EPSG:{epsg}'

        area_lakes = lakes_gdf[area_mask].copy()
        if str(area_lakes.crs) != utm_crs:
            area_lakes = area_lakes.to_crs(utm_crs)

        with rasterio.open(dem_path) as src:
            nodata = src.nodata if src.nodata is not None else -9999

            for row_idx, lake_row in tqdm(
                area_lakes.iterrows(),
                total=len(area_lakes),
                desc=f'  Dam features {area}',
            ):
                geom = lake_row.geometry
                try:
                    exterior = geom.exterior
                    n_pts = max(60, int(exterior.length / 30))
                    pts = [
                        exterior.interpolate(i / n_pts, normalized=True)
                        for i in range(n_pts)
                    ]

                    elevs = []
                    for pt in pts:
                        try:
                            r, c = src.index(pt.x, pt.y)
                            val = src.read(
                                1,
                                window=((r, r + 1), (c, c + 1))
                            )[0, 0]
                            if val != nodata:
                                elevs.append((pt, float(val)))
                        except Exception:
                            pass

                    if elevs:
                        dam_pt, dam_elev = min(elevs, key=lambda x: x[1])

                        lake_elev = lake_row.get(
                            'elev_mean',
                            float(np.mean([e[1] for e in elevs]))
                        )
                        freeboard = float(lake_elev) - dam_elev

                        # Sample 100 m downstream
                        ds_elevs = []
                        for angle in np.linspace(0, 2 * np.pi, 8, endpoint=False):
                            test_x = dam_pt.x + 100.0 * np.cos(angle)
                            test_y = dam_pt.y + 100.0 * np.sin(angle)
                            test_pt = Point(test_x, test_y)
                            if not geom.contains(test_pt):
                                try:
                                    r, c = src.index(test_x, test_y)
                                    ds_val = src.read(
                                        1, window=((r, r + 1), (c, c + 1))
                                    )[0, 0]
                                    if ds_val != nodata and ds_val < dam_elev:
                                        ds_elevs.append(float(ds_val))
                                except Exception:
                                    pass

                        dam_height = dam_elev - min(ds_elevs) if ds_elevs else np.nan

                        lakes_gdf.loc[row_idx, 'dam_elev'] = dam_elev
                        lakes_gdf.loc[row_idx, 'freeboard'] = freeboard
                        lakes_gdf.loc[row_idx, 'dam_height'] = dam_height
                except Exception:
                    pass

        print(f'  Dam features done: {area}')

    return lakes_gdf


if len(lakes_gdf) > 0:
    lakes_gdf = estimate_dam_features(lakes_gdf)

Estimating dam features...


  Dam features cordillera_blanca:   0%|          | 0/3083 [00:00<?, ?it/s]

  Dam features cordillera_blanca:   0%|          | 14/3083 [00:00<00:23, 130.64it/s]

  Dam features cordillera_blanca:   1%|          | 28/3083 [00:00<00:28, 107.62it/s]

  Dam features cordillera_blanca:   1%|▏         | 40/3083 [00:00<00:35, 86.47it/s] 

...

  Dam features carabaya:  66%|██████▌   | 57/87 [00:00<00:00, 94.17it/s]

  Dam features carabaya:  78%|███████▊  | 68/87 [00:00<00:00, 85.84it/s]

  Dam features carabaya:  93%|█████████▎| 81/87 [00:00<00:00, 97.18it/s]

  Dam features carabaya: 100%|██████████| 87/87 [00:00<00:00, 94.50it/s]

  Dam features done: carabaya


## 8. Temporal Growth Features

For each lake we compute MNDWI for all available Sentinel-2 years (2017–2025)
and fit a linear regression of detected area versus year.
The regression slope (m²/yr) is the primary growth rate feature.

**Key reference**: Shugar et al. (2020) Nature Climate Change show that
Andean glacial lake area increased by ~50% between 1990 and 2018.

In [9]:
def _ndwi_from_paths(green_path, nir_path, geom, buf=100.0):
    """
    Compute MNDWI-based water area for a single lake from band paths.

    Returns detected water area in m^2, or NaN on failure.
    """
    if not Path(green_path).exists() or not Path(nir_path).exists():
        return np.nan
    try:
        buf_geom = geom.buffer(buf)
        with rasterio.open(green_path) as g_src, \
             rasterio.open(nir_path) as n_src:
            g_img, g_tf = rio_mask(g_src, [buf_geom], crop=True)
            n_img, _ = rio_mask(n_src, [buf_geom], crop=True)
            green = g_img[0].astype(float)
            nir = n_img[0].astype(float)
            with np.errstate(divide='ignore', invalid='ignore'):
                ndwi = (green - nir) / (green + nir)
                ndwi = np.where(np.isfinite(ndwi), ndwi, 0.0)
            water = ndwi > 0.3
            px_area = abs(g_tf[0] * g_tf[4])  # m^2 per pixel
            return float(np.sum(water) * px_area)
    except Exception:
        return np.nan


def extract_growth_features(lakes_gdf):
    """
    Compute lake growth rate from multi-temporal Sentinel-2 imagery.

    For each available year, NDWI-based water area is detected within
    a 100 m buffer of the lake.  A linear regression (scipy.stats.linregress)
    of area vs. year gives growth_rate_m2_yr.

    Added columns:
      area_YYYY (per year), total_growth_m2, growth_rate_m2_yr,
      area_2017_est, area_2025_est

    Returns
    -------
    GeoDataFrame with growth columns appended.
    """
    print('Extracting temporal growth features...')

    area_col = 'area_name' if 'area_name' in lakes_gdf.columns else 'study_area'
    lakes_gdf['growth_rate_m2_yr'] = np.nan
    lakes_gdf['total_growth_m2'] = np.nan
    lakes_gdf['area_2017_est'] = np.nan
    lakes_gdf['area_2025_est'] = np.nan

    for area in STUDY_AREAS:
        area_mask = lakes_gdf[area_col] == area
        if area_mask.sum() == 0:
            continue

        s2_area_dir = RAW_DIR / 'sentinel2' / area
        if not s2_area_dir.exists():
            continue

        # Discover available years
        year_dirs = {}
        for d in s2_area_dir.iterdir():
            if d.is_dir():
                yr = None
                if d.name.startswith('S2') and len(d.name) > 15:
                    try:
                        yr = int(d.name.split('_')[2][:4])
                    except (IndexError, ValueError):
                        pass
                elif d.name.startswith('year_'):
                    try:
                        yr = int(d.name.replace('year_', ''))
                    except ValueError:
                        pass
                elif d.name.isdigit() and len(d.name) == 4:
                    yr = int(d.name)  # standard structure: area/YYYY/
                if yr is not None and yr not in year_dirs:
                    year_dirs[yr] = d

        valid_years = sorted(year_dirs.keys())
        if len(valid_years) < 2:
            print(f'  {area}: fewer than 2 years, skipping growth.')
            continue

        print(f'  {area}: years {valid_years}')

        epsg = CRS_LOOKUP.get(area, 32718)
        utm_crs = f'EPSG:{epsg}'
        area_lakes = lakes_gdf[area_mask].copy()
        if str(area_lakes.crs) != utm_crs:
            # Reproject to UTM to match Sentinel-2 native CRS
            area_lakes_utm = area_lakes.to_crs(utm_crs)
        else:
            area_lakes_utm = area_lakes

        # Year columns
        for yr in valid_years:
            col_name = f'area_{yr}'
            if col_name not in lakes_gdf.columns:
                lakes_gdf[col_name] = np.nan

        for row_idx, lake_row in area_lakes_utm.iterrows():
            geom = lake_row.geometry
            year_areas = {}

            for yr, yr_dir in year_dirs.items():
                green_tif = list(yr_dir.glob('*B03*.tif'))
                nir_tif = list(yr_dir.glob('*B08*.tif'))
                if green_tif and nir_tif:
                    a = _ndwi_from_paths(green_tif[0], nir_tif[0], geom)
                    if not np.isnan(a):
                        year_areas[yr] = a
                        lakes_gdf.loc[row_idx, f'area_{yr}'] = a

            if len(year_areas) >= 2:
                yrs_arr = np.array(sorted(year_areas.keys()))
                areas_arr = np.array([year_areas[y] for y in yrs_arr])
                slope, intercept, _, _, _ = linregress(yrs_arr, areas_arr)
                lakes_gdf.loc[row_idx, 'growth_rate_m2_yr'] = slope
                lakes_gdf.loc[row_idx, 'total_growth_m2'] = areas_arr[-1] - areas_arr[0]
                lakes_gdf.loc[row_idx, 'area_2017_est'] = intercept + slope * 2017
                lakes_gdf.loc[row_idx, 'area_2025_est'] = intercept + slope * 2025

        print(f'  Growth features done: {area}')

    return lakes_gdf


if len(lakes_gdf) > 0:
    lakes_gdf = extract_growth_features(lakes_gdf)

Extracting temporal growth features...
  cordillera_blanca: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


  Growth features done: cordillera_blanca
  cordillera_vilcanota: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


  Growth features done: cordillera_vilcanota
  cordillera_central: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


  Growth features done: cordillera_central
  chile_andes_centrales: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


  Growth features done: chile_andes_centrales
  cordillera_raura: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


...

  Growth features done: ecuador_antisana
  bolivia_cordillera_real: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


  Growth features done: bolivia_cordillera_real
  patagonia_sur: years [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


  Growth features done: patagonia_sur
  apolobamba: years [2017, 2019, 2021, 2023, 2024]


  Growth features done: apolobamba
  carabaya: years [2017, 2018, 2019, 2020, 2021, 2023, 2024, 2025]


  Growth features done: carabaya


## 9. Depth and Volume Estimation

Four empirical area-depth power laws are applied and averaged.
A regional correction factor of 1.15 is applied for the Tropical Andes
(Emmer 2017; Mergili et al. 2015).

Volume uses the paraboloid shape factor 0.45 (Huggel et al. 2002).

In [10]:
def estimate_lake_depth(area_m2):
    """
    Empirical area-depth estimates using four published relationships.

    Methods
    -------
    Cook & Quincey (2015)  : D = 0.104 * A^0.42  [Himalaya]
    Huggel et al. (2002)   : D = 0.104 * A^0.42  [Alps/Global]
    Yao et al. (2012)      : D = 0.148 * A^0.410 [Tibet]
    O'Connor et al. (2001) : D = 0.133 * A^0.424 [Global]

    A Tropical Andes correction factor of 1.15 is applied to the ensemble mean.

    Parameters
    ----------
    area_m2 : float or array-like  (lake area in m^2)

    Returns
    -------
    dict with individual method estimates, ensemble statistics,
    and Andes-corrected mean depth.
    """
    a = np.asarray(area_m2, dtype=float)

    methods = {
        'cook_quincey_2015': 0.104 * (a ** 0.420),
        'huggel_2002':       0.104 * (a ** 0.420),
        'yao_2012':          0.148 * (a ** 0.410),
        'oconnor_2001':      0.133 * (a ** 0.424),
    }
    stack = np.vstack(list(methods.values()))

    ens_mean = np.mean(stack, axis=0)
    ens_std = np.std(stack, axis=0)
    andes = ens_mean * ANDES_DEPTH_CORRECTION

    return {
        'methods': methods,
        'ensemble_mean': ens_mean,
        'ensemble_std': ens_std,
        'depth_m': andes,  # primary estimate
        'depth_std_m': ens_std * ANDES_DEPTH_CORRECTION,
    }


def estimate_volume(area_m2, depth_m, shape_factor=0.45):
    """
    Lake volume using paraboloid shape approximation.

    V = shape_factor * A * D_max
    D_mean ~ 0.6 * D_max for moraine-dammed lakes.

    Parameters
    ----------
    area_m2      : float  (m^2)
    depth_m      : float  (mean depth in m)
    shape_factor : float  (default 0.45, Huggel et al. 2002)

    Returns
    -------
    float : volume in m^3
    """
    d_max_est = depth_m / 0.6
    return shape_factor * area_m2 * d_max_est


if len(lakes_gdf) > 0:
    print('Applying multi-method depth estimation...')

    depth_res = estimate_lake_depth(lakes_gdf['area_m2'].values)

    for method_name, vals in depth_res['methods'].items():
        lakes_gdf[f'depth_{method_name}'] = vals

    lakes_gdf['depth_ensemble_mean'] = depth_res['ensemble_mean']
    lakes_gdf['depth_ensemble_std'] = depth_res['ensemble_std']
    lakes_gdf['depth_m'] = depth_res['depth_m']         # Andes-corrected
    lakes_gdf['depth_std_m'] = depth_res['depth_std_m']

    lakes_gdf['volume_m3'] = estimate_volume(
        lakes_gdf['area_m2'].values,
        lakes_gdf['depth_m'].values
    )
    lakes_gdf['area_depth_ratio'] = lakes_gdf['area_m2'] / lakes_gdf['depth_m'].replace(0, np.nan)

    print(f'  Mean depth (Andes-corrected): {lakes_gdf["depth_m"].mean():.1f} m')
    print(f'  Total volume: {lakes_gdf["volume_m3"].sum()/1e6:.2f} million m\u00b3')
    print(f'  Mean A/D ratio: {lakes_gdf["area_depth_ratio"].mean():.0f}')

Applying multi-method depth estimation...
  Mean depth (Andes-corrected): 10.1 m
  Total volume: 2783967.75 million m³
  Mean A/D ratio: 4151


## 10. Risk Indicators

Composite indicators combining physical size, topographic hazard,
and dynamic growth signal.

In [11]:
def calculate_risk_indicators(lakes_gdf):
    """
    Derive composite risk indicators for GLOF susceptibility.

    Added columns
    -------------
    potential_energy  : volume_m3 * 9.81 * dam_height (proxy in J)
    size_class        : small / medium / large (area thresholds)
    risk_score        : normalised composite (area_depth_ratio +
                        slope_mean + growth_rate_m2_yr)

    Returns
    -------
    GeoDataFrame with risk columns appended.
    """
    print('Calculating risk indicators...')

    # Potential energy: gravitational potential energy proxy
    if 'volume_m3' in lakes_gdf.columns and 'dam_height' in lakes_gdf.columns:
        lakes_gdf['potential_energy'] = (
            lakes_gdf['volume_m3'] * 9.81 * lakes_gdf['dam_height']
        )

    # Size class (area in m^2 thresholds)
    lakes_gdf['size_class'] = pd.cut(
        lakes_gdf['area_m2'],
        bins=[0, 10_000, 1_000_000, np.inf],
        labels=['small', 'medium', 'large'],
    )

    # Normalised risk score (0-1 scale per component)
    def _norm(series):
        mn, mx = series.min(), series.max()
        return (series - mn) / (mx - mn) if mx > mn else series * 0.0

    score = pd.Series(0.0, index=lakes_gdf.index)
    weights = {
        'area_depth_ratio': 0.35,
        'slope_mean': 0.35,
        'growth_rate_m2_yr': 0.30,
    }
    for col, w in weights.items():
        if col in lakes_gdf.columns:
            score += w * _norm(lakes_gdf[col].fillna(0))

    lakes_gdf['risk_score'] = score

    print('  Risk indicators calculated.')
    return lakes_gdf


if len(lakes_gdf) > 0:
    lakes_gdf = calculate_risk_indicators(lakes_gdf)

Calculating risk indicators...
  Risk indicators calculated.


## 11. Save Feature Dataset

In [12]:
if len(lakes_gdf) > 0:
    out_dir = PROCESSED_DIR / 'features'
    out_dir.mkdir(parents=True, exist_ok=True)

    gpkg_path = out_dir / 'lake_features.gpkg'
    csv_path = out_dir / 'lake_features.csv'

    # Drop categorical columns that may cause issues in some GPKG drivers
    save_gdf = lakes_gdf.copy()
    for col in save_gdf.select_dtypes(include='category').columns:
        save_gdf[col] = save_gdf[col].astype(str)

    save_gdf.to_file(gpkg_path, driver='GPKG')
    print(f'GeoPackage saved: {gpkg_path}')

    csv_df = save_gdf.drop(columns=['geometry'])
    csv_df.to_csv(csv_path, index=False)
    print(f'CSV saved       : {csv_path}')

    # Feature metadata
    meta = {
        'n_lakes': int(len(lakes_gdf)),
        'n_features': int(len(lakes_gdf.columns) - 1),
        'feature_names': [c for c in lakes_gdf.columns if c != 'geometry'],
        'study_areas': lakes_gdf['area_name'].unique().tolist()
        if 'area_name' in lakes_gdf.columns else [],
    }
    with open(out_dir / 'feature_metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'\nFeature extraction complete.')
    print(f'  Lakes       : {len(lakes_gdf)}')
    print(f'  Features    : {len(lakes_gdf.columns) - 1}')
else:
    print('No lakes to save.')

GeoPackage saved: D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\features\lake_features.gpkg


CSV saved       : D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\features\lake_features.csv

Feature extraction complete.
  Lakes       : 12700
  Features    : 49


## 12. Feature Correlation Heatmap

In [13]:
if len(lakes_gdf) > 0:
    # Select a representative subset of features for the figure
    key_features = [
        'area_m2', 'depth_m', 'volume_m3', 'area_depth_ratio',
        'perimeter_m', 'compactness', 'elongation', 'convexity',
        'slope_mean', 'slope_max', 'tri_mean',
        'dam_height', 'freeboard',
        'growth_rate_m2_yr', 'risk_score',
    ]
    plot_cols = [c for c in key_features if c in lakes_gdf.columns]
    corr = lakes_gdf[plot_cols].corr()

    fig, ax = plt.subplots(figsize=(len(plot_cols) * 0.65 + 2,
                                    len(plot_cols) * 0.65 + 2))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Pearson r')

    ax.set_xticks(range(len(plot_cols)))
    ax.set_yticks(range(len(plot_cols)))
    ax.set_xticklabels(plot_cols, rotation=60, ha='right', fontsize=9)
    ax.set_yticklabels(plot_cols, fontsize=9)

    # Annotate cells
    for i in range(len(plot_cols)):
        for j in range(len(plot_cols)):
            val = corr.values[i, j]
            color = 'white' if abs(val) > 0.6 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=color)

    ax.set_title('Feature Correlation Matrix (key features)',
                 fontsize=13, fontweight='bold', pad=12)
    plt.tight_layout()

    fig_dir = project_root / 'figures'
    fig_dir.mkdir(exist_ok=True)
    fig.savefig(fig_dir / 'feature_correlation_heatmap.png',
                dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Correlation heatmap saved.')
else:
    print('No lakes to correlate.')

Correlation heatmap saved.


## 13. Feature Importance Preview

Quick Random Forest run to rank feature importance before full model training.
Note: at this stage labels come from the historical GLOF database (NB14),
so this cell loads the labeled dataset if it already exists.

In [14]:
labeled_path = PROCESSED_DIR / 'labeled' / 'training_data.csv'

if labeled_path.exists():
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler

    train_df = pd.read_csv(labeled_path)
    if 'had_glof' not in train_df.columns:
        print('No label column found. Run NB14 first for labeled data.')
    else:
        exclude_cols = {
            'had_glof', 'study_area', 'area_name', 'lake_id',
            'scene_date', 'detection_date', 'year',
        }
        feat_cols = [
            c for c in train_df.columns
            if c not in exclude_cols
            and train_df[c].dtype in ('float64', 'int64', 'float32', 'int32')
        ]

        X = train_df[feat_cols].fillna(train_df[feat_cols].median())
        y = train_df['had_glof']

        rf = RandomForestClassifier(
            n_estimators=200, random_state=42,
            class_weight='balanced', n_jobs=-1
        )
        rf.fit(X, y)

        importance_df = (
            pd.DataFrame({'feature': feat_cols,
                          'importance': rf.feature_importances_})
            .sort_values('importance', ascending=False)
            .head(20)
            .reset_index(drop=True)
        )

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(importance_df['feature'][::-1],
                importance_df['importance'][::-1],
                color='steelblue', edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Mean Decrease in Impurity')
        ax.set_title('Top-20 Feature Importances (Random Forest, quick preview)',
                     fontweight='bold')
        ax.grid(True, axis='x', alpha=0.3)
        plt.tight_layout()

        fig_dir = project_root / 'figures'
        fig_dir.mkdir(exist_ok=True)
        fig.savefig(fig_dir / 'feature_importance_preview.png',
                    dpi=150, bbox_inches='tight')
        plt.close(fig)

        print('Top-10 features:')
        print(importance_df.head(10).to_string(index=False))
else:
    print(f'Labeled data not found: {labeled_path}')
    print('Run NB14 (historical GLOFs) first to generate training labels.')

Top-10 features:
          feature  importance
glof_match_dist_m    0.131151
   glof_volume_m3    0.128753
       risk_score    0.067150
        slope_std    0.039895
        slope_max    0.037732
       elongation    0.033809
          area_m2    0.030766
         tri_mean    0.030143
         elev_max    0.029307
      perimeter_m    0.027768
